In [ ]:
#@title Cell 43.1 — Overview and BMC-AT model
from IPython.display import Markdown, display

display(Markdown(r"""
# NB43 — BMC-AT Bayesian calibration and pathogen-out validation (v0.1)

**Objective:** Apply **Bayesian Model C — two-piece asymmetric Student t (nu=4) (BMC-AT)** to all 26 antibiotics, using the exact training-only cross-fitted predictions from NB40, and evaluate on the original 50,460 independently held-out Model C predictions.

For antibiotic $a$ let $e_i=y_i-r_i$ be observed minus Model C log₂(MIC). BMC-AT fits the normalised **two-piece Student t** density: $f(e)=2\,t_4((e-b)/\sigma_{\mathrm{side}})/(\sigma_L+\sigma_R)$ where $\sigma_L=\sigma e^{-\lambda}$ for $e<b$ and $\sigma_R=\sigma e^{\lambda}$ for $e\ge b$. Priors: $b\sim N(0,1)$, $\sigma\sim\operatorname{Exponential}(0.5)$, $\lambda\sim N(0,0.5^2)$; $\nu=4$ is **fixed**. $b$ is the split point (mode), not necessarily the predictive mean or median. At $\lambda=0$ this is exactly BMC-T. Negative and positive $\lambda$ are both permitted. No chromosomal MIC score is used.

**Posterior computation:** MAP-centred Sobol importance quadrature in $(b,\log\sigma,\lambda)$ with exact likelihood/prior reweighting; reported predictive quantiles are computed by inversion of the exact posterior-weighted two-piece-t CDF. Effective sample sizes are checked; the posterior calculation uses training-only MIC residuals. No MCMC, so no R-hat applies.

**Independent evaluation:** BMC parameters for outer fold $k$ use only NB40's inner out-of-fold Model C residuals for outer-training pathogens; the original outer test pathogens' observed MICs are reserved for evaluation. The five calibration datasets overlap and must not be pooled as independent observations. No common resistance breakpoint is assumed across antibiotics.

**Figures:** (1) 95% coverage across all 26 antibiotics, (2) observed MICs above upper 95% predictive limit by antibiotic, (3) individual BioSamples' intervals for four focus antibiotics (ceftazidime, cefepime, ciprofloxacin, meropenem). The third plot uses the 25 largest original Model C underprediction cases per antibiotic, in an identical order across models for meaningful comparison.

**Inputs:** Two NB40 output files only. NB43 also reads NB41 and NB42 held-out prediction outputs for the final three-model comparison. **Outputs:** `RefMIC/03_Results/43_BMC_AT_Pathogen_Out_Validation`. Run all cells **43.1–43.9** in order. No original source files are modified.

**Caution:** The original Model C kernels use cohort-wide genomic information (transductive construction); model settings were selected within each outer training fold. NB43's evaluation is held-out **for MIC outcomes**, under those same Model C design choices.
"""))


In [ ]:
#@title Cell 43.2 — Locate NB40 outputs automatically
import os, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.special import logsumexp, ndtr
from scipy.optimize import brentq
import matplotlib.pyplot as plt
from IPython.display import display

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
except ImportError:
    pass

ROOT=Path(os.environ.get('BMC_PROJECT_ROOT','/content/drive/MyDrive/RefMIC'))
if not ROOT.is_dir():
    raise FileNotFoundError(f'RefMIC project folder not found at {ROOT}; set BMC_PROJECT_ROOT if it is elsewhere.')
INPUT_DIR=ROOT/'03_Results'/'40_BMC_Training_Only_Calibration'
OUT=ROOT/'03_Results'/'43_BMC_AT_Pathogen_Out_Validation'
OUT.mkdir(parents=True,exist_ok=True)

CALIB_NAME='40_BMC_training_only_inner_predictions_all_outer_folds.csv.gz'
TEST_NAME='40_BMC_independent_outer_test_predictions.csv.gz'

def get_file(name):
    direct=INPUT_DIR/name
    if direct.is_file(): return direct
    searchroot=Path('/content/drive') if Path('/content/drive').is_dir() else ROOT
    matches=[]
    for directory,dirs,files in os.walk(searchroot):
        dirs[:]=[x for x in dirs if x not in {'.Trashes','.Trash-1000','.git','__pycache__','node_modules'}]
        if name in files: matches.append(Path(directory)/name)
    if not matches:
        raise FileNotFoundError(f'Could not find {name} under {searchroot}. Please run NB40 Cell 40.7.')
    if len(matches)>1:
        def sha(p):
            h=hashlib.sha256()
            with open(p,'rb') as f:
                for buf in iter(lambda:f.read(4*1024*1024),b''): h.update(buf)
            return h.hexdigest()
        if len({sha(p) for p in matches})!=1:
            raise ValueError(f'Different copies of {name} found; unable to choose:\n'+ '\n'.join(map(str,matches)))
        print('Multiple byte-identical copies found for',name)
    return sorted(matches,key=lambda p:(0 if ROOT in p.parents else 1,len(p.parts),str(p)))[0]

CALIB_PATH=get_file(CALIB_NAME)
TEST_PATH=get_file(TEST_NAME)
print('Training-only calibration:',CALIB_PATH)
print('Independent held-out test:',TEST_PATH)
print('NB43 results:',OUT)


In [ ]:
#@title Cell 43.3 — Load inputs and verify fold separation
calib=pd.read_csv(CALIB_PATH,compression='gzip')
outer_test=pd.read_csv(TEST_PATH,compression='gzip')

calib_cols={'outer_fold_for_calibration','inner_fold','interaction_row','biosample','antibiotic','observed_log2_mic','inner_oof_predicted_log2_mic','training_residual_log2'}
test_cols={'outer_fold','interaction_row','biosample','antibiotic','observed_log2_mic','model_c_outer_test_predicted_log2_mic'}
for name,tab,needed in [('calibration',calib,calib_cols),('outer test',outer_test,test_cols)]:
    if not needed.issubset(tab.columns):
        raise ValueError(f'{name}: missing columns {sorted(needed-set(tab.columns))}')

if len(calib)!=201840 or len(outer_test)!=50460:
    raise ValueError('NB40 data sizes do not match 201,840 training records and 50,460 outer test records.')
if outer_test.biosample.nunique()!=9058 or outer_test.antibiotic.nunique()!=26:
    raise ValueError('Unexpected number of BioSamples or antibiotics.')
if outer_test.interaction_row.duplicated().any() or outer_test.duplicated(['biosample','antibiotic']).any():
    raise ValueError('Duplicate independent Model C interactions.')
if (outer_test.groupby('biosample').outer_fold.nunique()!=1).any():
    raise ValueError('Outer-test BioSample appears in more than one fold.')
if set(outer_test.outer_fold.unique())!=set(range(1,6)) or set(calib.outer_fold_for_calibration.unique())!=set(range(1,6)):
    raise ValueError('Expected exactly five matching outer folds.')
if calib.duplicated(['outer_fold_for_calibration','interaction_row']).any():
    raise ValueError('Duplicate calibration records within an outer fold.')
if (calib.groupby(['outer_fold_for_calibration','biosample']).inner_fold.nunique()!=1).any():
    raise ValueError('A calibration BioSample appears in more than one inner fold.')

original=outer_test.set_index('interaction_row')[['biosample','antibiotic','observed_log2_mic','outer_fold']]
for fold in range(1,6):
    cal=calib.loc[calib.outer_fold_for_calibration.eq(fold)]
    test=outer_test.loc[outer_test.outer_fold.eq(fold)]
    if len(cal)!=len(outer_test)-len(test):raise ValueError(f'Fold {fold} wrong calibration size.')
    if set(cal.interaction_row)!=set(outer_test.interaction_row)-set(test.interaction_row):
        raise ValueError(f'Fold {fold} calibration interactions do not match outer training set.')
    if set(cal.biosample)&set(test.biosample):raise ValueError(f'Fold {fold} test/calibration BioSample overlap.')
    check=original.loc[cal.interaction_row.to_numpy()]
    if not np.array_equal(check.biosample.astype(str).to_numpy(),cal.biosample.astype(str).to_numpy()):
        raise ValueError(f'Fold {fold} BioSample mismatch.')
    if not np.array_equal(check.antibiotic.astype(str).to_numpy(),cal.antibiotic.astype(str).to_numpy()):
        raise ValueError(f'Fold {fold} antibiotic mismatch.')
    if not np.allclose(check.observed_log2_mic.to_numpy(),cal.observed_log2_mic.to_numpy(),atol=1e-9,rtol=0):
        raise ValueError(f'Fold {fold} observed MIC mismatch.')

for tab,nums in [(calib,['observed_log2_mic','inner_oof_predicted_log2_mic','training_residual_log2']),
                 (outer_test,['observed_log2_mic','model_c_outer_test_predicted_log2_mic'])]:
    if not np.isfinite(tab[nums].to_numpy(dtype=float)).all():raise ValueError('Non-finite MIC values/predictions.')
if not np.allclose(calib.training_residual_log2,calib.observed_log2_mic-calib.inner_oof_predicted_log2_mic,rtol=0,atol=1e-8):
    raise ValueError('NB40 residuals are inconsistent with observed minus predicted MIC.')

print(f'Validated: {len(calib):,} overlapping training-only records across five outer fits.')
print(f'Validated: {len(outer_test):,} independent held-out records; {outer_test.biosample.nunique():,} BioSamples; 26 antibiotics.')
print('All five folds: calibration/test BioSamples disjoint and observation values identical.')
display(outer_test.groupby('outer_fold',as_index=False).agg(test_rows=('interaction_row','size'),test_pathogens=('biosample','nunique')))


In [ ]:
#@title Cell 43.4 — Fit independent training-only Bayesian AT posteriors
# Posterior importance quadrature (Sobol draws, exactly weighted), no MCMC.
# Target = two- or three-parameter posterior in (b, log(sigma) [, lambda]).
# Integrals for predictive quantiles are calculated from the weighted Student-t CDF.
# Checkpoints allow restarts without refitting completed outer folds.
from scipy.optimize import minimize, brentq
from scipy.special import gammaln, ndtri, logsumexp
from scipy.stats import t as student_t
from scipy.stats import qmc
import hashlib, json

NU=4.0
PRIOR_SIGMA_RATE=0.5
PRIOR_LAMBDA_SD=0.5
PARAM_3=3
BASE_SOBOL_POWER=12        # 4096 quasi-random posterior proposal points per fit
RETRY_SOBOL_POWER=13       # 8192 points when importance ESS is low
CHECKPOINT_VERSION='43_AT_posterior_v01_sobol2stage'
LOG_T_CONSTANT=float(gammaln((NU+1)/2)-gammaln(NU/2)-0.5*np.log(NU*np.pi))

# NB40 calibration files are fixed when NB41/42/43 are compared.
def file_sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for c in iter(lambda:f.read(4*1024*1024),b''):h.update(c)
    return h.hexdigest()
INPUT_SHA256=file_sha256(CALIB_PATH)

def log_target_batched(theta,errors,batch_size=128):
    "Exact log posterior density for transformed parameters (b,eta[,lambda]); constants omitted."
    e=np.asarray(errors,np.float64)
    theta=np.asarray(theta,np.float64)
    output=np.full(len(theta),-np.inf,dtype=float)
    for start in range(0,len(theta),batch_size):
        batch=theta[start:start+batch_size]
        b=batch[:,0]
        eta=batch[:,1]
        lam=batch[:,2] if PARAM_3==3 else np.zeros(len(batch))
        valid=(np.abs(b)<=15)&(eta>=-7)&(eta<=4)&(np.abs(lam)<=3)
        if not valid.any():continue
        ids=np.flatnonzero(valid)
        bb=b[ids,None]; ee=eta[ids,None]; ll=lam[ids,None]
        if PARAM_3==3:
            side_log_scale=np.where(e[None,:]<bb,ee-ll,ee+ll)
            z=(e[None,:]-bb)*np.exp(-side_log_scale)
            log_norm=np.log(2)-np.logaddexp(ee-ll,ee+ll)
        else:
            z=(e[None,:]-bb)*np.exp(-ee)
            log_norm=-ee
        log_lik=len(e)*LOG_T_CONSTANT+len(e)*log_norm[:,0] \
            -0.5*(NU+1)*np.log1p((z*z)/NU).sum(axis=1)
        prior=(-0.5*b[ids]**2-PRIOR_SIGMA_RATE*np.exp(eta[ids])+eta[ids])
        if PARAM_3==3:prior=prior-0.5*(lam[ids]/PRIOR_LAMBDA_SD)**2
        output[start+ids]=log_lik+prior
    return output

def hessian(fun,x,eps=0.003):
    x=np.asarray(x,dtype=float)
    k=len(x)
    E=np.eye(k)*eps
    h=np.zeros((k,k),float)
    f0=float(fun(x))
    for i in range(k):
        h[i,i]=(fun(x+E[i])-2*f0+fun(x-E[i]))/(eps*eps)
        for j in range(i+1,k):
            h[i,j]=h[j,i]=(fun(x+E[i]+E[j])-fun(x+E[i]-E[j]) \
                  -fun(x-E[i]+E[j])+fun(x-E[i]-E[j]))/(4*eps*eps)
    return h

def weighted_quantile(x,w,probs):
    idx=np.argsort(x)
    vals=np.asarray(x)[idx];ws=np.asarray(w)[idx]
    return np.interp(probs,np.cumsum(ws),vals)

def predictive_cdf(x,theta,w):
    b=theta[:,0];sigma=np.exp(theta[:,1]);lam=theta[:,2] if PARAM_3==3 else np.zeros(len(theta))
    if PARAM_3==2:
        cdf=student_t.cdf((x-b)/sigma,df=NU)
    else:
        left=sigma*np.exp(-lam);right=sigma*np.exp(lam)
        mass_left=left/(left+right);mass_right=1-mass_left
        # Exact normalised two-piece t CDF, including the mode split-point b.
        cdf=np.where(x<b,2*mass_left*student_t.cdf((x-b)/left,df=NU),
                     1-2*mass_right*student_t.sf((x-b)/right,df=NU))
    return float(np.dot(w,cdf))

def fit_posterior(errors,seed):
    e=np.asarray(errors,np.float64)
    if len(e)<15 or not np.isfinite(e).all():raise ValueError('Insufficient/nonfinite training residuals.')
    med=float(np.median(e));sd=float(np.std(e))
    initial=[med,np.log(max(0.15,sd*0.8))]
    if PARAM_3==3:initial.append(0.0)
    def neg_target(v):return -log_target_batched(np.asarray(v)[None,:],e)[0]
    bounds=[(-12,12),(-6,3)] + ([(-2.0,2.0)] if PARAM_3==3 else [])
    fit=minimize(neg_target,np.asarray(initial),method='L-BFGS-B',bounds=bounds,
                 options={'maxiter':450,'ftol':1e-11})
    if not np.isfinite(fit.fun):raise RuntimeError('Posterior mode optimisation failed.')
    mode=np.asarray(fit.x)
    h=hessian(neg_target,mode)
    eigvals,Q=np.linalg.eigh(h)
    if np.min(eigvals)<=0:
        raise RuntimeError(f'Posterior curvature is not positive definite at mode: {eigvals}.')
    sqrt_cov=(Q / np.sqrt(eigvals))@Q.T
    candidates=[]
    for power,scale in [(BASE_SOBOL_POWER,1.32),(RETRY_SOBOL_POWER,1.15)]:
        # Sobol avoids ordinary Monte Carlo's excessive fluctuation for 130 small posterior fits.
        U=qmc.Sobol(d=PARAM_3,scramble=True,seed=seed).random_base2(m=power)
        Z=ndtri(np.clip(U,1e-12,1-1e-12))
        theta=mode[None,:]+(Z @ sqrt_cov.T)*scale
        logp=log_target_batched(theta,e)
        # Log density of proposal N(mode, scale² * inv(H)), ignoring universal constants.
        logq=-0.5*np.square(Z).sum(axis=1) - PARAM_3*np.log(scale) \
              +0.5*np.log(eigvals).sum()
        lw=logp-logq
        if not np.isfinite(lw).any():raise RuntimeError('Importance sampling has no finite draws.')
        weights=np.exp(lw-logsumexp(lw))
        ess=float(1/np.sum(weights**2))
        candidates.append((ess,theta,weights,power,scale))
        if ess>=max(350,0.13*len(theta)):break
    ess,theta,w,power,scale=max(candidates,key=lambda z:z[0])
    if ess<350:
        raise RuntimeError(f'Importance effective sample size only {ess:.0f}; posterior quadrature unreliable.')

    b=theta[:,0];sigma=np.exp(theta[:,1]);lam=theta[:,2] if PARAM_3==3 else np.zeros(len(theta))
    splus=sigma*np.exp(lam);sminus=sigma*np.exp(-lam)
    # Bracket the predictive CDF. With a t4 distribution 50 scales are ample;
    # expand automatically rather than silently returning a boundary quantile.
    wide=float(max(10,50*np.max(splus),50*np.max(sminus)))
    lo=float(np.min(b)-wide);hi=float(np.max(b)+wide)
    qpred=[]
    for q in (.025,.5,.975):
        while predictive_cdf(lo,theta,w)>q:lo-=wide
        while predictive_cdf(hi,theta,w)<q:hi+=wide
        qpred.append(float(brentq(lambda x:predictive_cdf(x,theta,w)-q,lo,hi,xtol=1e-9)))
    bq=weighted_quantile(b,w,[.025,.5,.975])
    sq=weighted_quantile(sigma,w,[.025,.5,.975])
    lq=weighted_quantile(lam,w,[.025,.5,.975])
    return {
        'n_calibration':len(e),'nu_fixed':NU,
        'posterior_mean_b_log2':float(np.dot(w,b)),
        'posterior_median_b_log2':float(bq[1]),
        'posterior_b_2p5pct':float(bq[0]),'posterior_b_97p5pct':float(bq[2]),
        'posterior_mean_sigma_log2':float(np.dot(w,sigma)),
        'posterior_median_sigma_log2':float(sq[1]),
        'posterior_sigma_2p5pct':float(sq[0]),'posterior_sigma_97p5pct':float(sq[2]),
        'posterior_mean_lambda':float(np.dot(w,lam)) if PARAM_3==3 else np.nan,
        'posterior_median_lambda':float(lq[1]) if PARAM_3==3 else np.nan,
        'posterior_lambda_2p5pct':float(lq[0]) if PARAM_3==3 else np.nan,
        'posterior_lambda_97p5pct':float(lq[2]) if PARAM_3==3 else np.nan,
        'posterior_p_lambda_gt_0':float(np.sum(w[lam>0])) if PARAM_3==3 else np.nan,
        'posterior_median_left_scale_log2':float(weighted_quantile(sminus,w,.5)),
        'posterior_median_right_scale_log2':float(weighted_quantile(splus,w,.5)),
        'predictive_residual_lower95_log2':qpred[0],
        'predictive_residual_median_log2':qpred[1],
        'predictive_residual_upper95_log2':qpred[2],
        'importance_ess':ess,'importance_ess_fraction':ess/len(w),
        'importance_max_weight':float(np.max(w)),
        'importance_points':len(w),
        'map_b':float(mode[0]),'map_log_sigma':float(mode[1]),
        'map_lambda':float(mode[2]) if PARAM_3==3 else np.nan,
    }

# Verify density: two-piece t must become t4 exactly for lambda=0; always integrates to one.
if PARAM_3==3:
    from scipy.integrate import quad
    echeck=np.array([-4.0,-1.0,.0,.8,2.0,5.0]);thetas=np.column_stack([
        np.repeat(0.3,len(echeck)),np.repeat(np.log(1.2),len(echeck)),np.zeros(len(echeck))])
    # direct normalisation by numeric quadrature of the two-piece definition:
    for lam in (-.4,0.,.4):
        ll=1.2*np.exp(-lam);rr=1.2*np.exp(lam)
        mass=quad(lambda x:2/(ll+rr)*student_t.pdf((x-.3)/(ll if x<.3 else rr),df=4),
                  -np.inf,np.inf,epsabs=1e-9)[0]
        assert abs(mass-1)<1e-8
    print('Two-piece Student t density: normalisation checks passed.')

post_rows=[]
for fold in range(1,6):
    chk=OUT/f'43_BMC_AT_training_only_posterior_outer_fold_{fold:02d}.csv'
    meta=OUT/f'43_BMC_AT_training_only_posterior_outer_fold_{fold:02d}.json'
    marker={'version':CHECKPOINT_VERSION,'input_sha256':INPUT_SHA256,'outer_fold':fold}
    valid=False
    if chk.is_file() and meta.is_file():
        try:
            saved=json.loads(meta.read_text())
            tab=pd.read_csv(chk)
            valid=(saved==marker and len(tab)==26 and tab.antibiotic.nunique()==26
                   and set(tab.antibiotic)==set(calib.antibiotic)
                   and tab.outer_fold.eq(fold).all()
                   and np.isfinite(tab[['predictive_residual_lower95_log2','predictive_residual_upper95_log2']]).all().all())
        except Exception:valid=False
    if not valid:
        rows=[]
        for k,(ab,g) in enumerate(calib.loc[calib.outer_fold_for_calibration.eq(fold)].groupby('antibiotic',sort=True)):
            row={'outer_fold':fold,'antibiotic':str(ab),
                 **fit_posterior(g.training_residual_log2.to_numpy(),seed=20261111+1000*fold+37*k)}
            rows.append(row)
            if (k+1)%5==0 or k==25:print(f'Fold {fold}: {k+1}/26 antibiotics fitted')
        tab=pd.DataFrame(rows)
        if len(tab)!=26:raise RuntimeError('Incomplete posterior fold; no checkpoint saved.')
        temporary=chk.with_suffix('.partial.csv')
        tab.to_csv(temporary,index=False);temporary.replace(chk)
        meta.write_text(json.dumps(marker,indent=2))
    else:print(f'Fold {fold}: reusing validated posterior checkpoint')
    post_rows.extend(tab.to_dict('records'))
posterior_summary=pd.DataFrame(post_rows).sort_values(['outer_fold','antibiotic']).reset_index(drop=True)
assert len(posterior_summary)==130
assert posterior_summary.importance_ess.min()>=350
print(f'BMC-AT: completed 130 independent antibiotic × outer-fold posteriors.')
print('Minimum importance-sampling effective sample size:',round(posterior_summary.importance_ess.min(),1))
cols=['outer_fold','antibiotic','n_calibration','posterior_mean_b_log2','posterior_median_sigma_log2','importance_ess']
if PARAM_3==3:cols+=['posterior_median_lambda','posterior_p_lambda_gt_0']
display(posterior_summary[cols].head(10).round(4))


In [ ]:
#@title Cell 43.5 — Evaluate independent outer-test predictions and summarise 26 antibiotics
shifts=posterior_summary[['outer_fold','antibiotic','predictive_residual_lower95_log2',
                          'predictive_residual_median_log2','predictive_residual_upper95_log2']]
bmc=outer_test.merge(shifts,on=['outer_fold','antibiotic'],how='left',validate='many_to_one',indicator=True)
if len(bmc)!=50460 or not bmc._merge.eq('both').all():
    raise ValueError('Some outer test predictions lack matching Bayesian parameters.')
bmc=bmc.drop(columns=['_merge'])
r=bmc.model_c_outer_test_predicted_log2_mic.to_numpy()
bmc['bmc_at_lower95_log2']=r+bmc.predictive_residual_lower95_log2
bmc['bmc_at_median_log2']=r+bmc.predictive_residual_median_log2
bmc['bmc_at_upper95_log2']=r+bmc.predictive_residual_upper95_log2
bmc=bmc.drop(columns=['predictive_residual_lower95_log2','predictive_residual_median_log2','predictive_residual_upper95_log2'])
bmc['model_c_residual_log2']=bmc.observed_log2_mic-bmc.model_c_outer_test_predicted_log2_mic
bmc['bmc_at_residual_log2']=bmc.observed_log2_mic-bmc.bmc_at_median_log2
bmc['covered_95pct']=(bmc.observed_log2_mic>=bmc.bmc_at_lower95_log2)&(bmc.observed_log2_mic<=bmc.bmc_at_upper95_log2)
bmc['above_upper95']=bmc.observed_log2_mic>bmc.bmc_at_upper95_log2
bmc['below_lower95']=bmc.observed_log2_mic<bmc.bmc_at_lower95_log2
bmc['distance_above_upper_log2']=np.maximum(0,bmc.observed_log2_mic-bmc.bmc_at_upper95_log2)
bmc['model_c_underprediction_ge2']=bmc.model_c_residual_log2>=2
bmc['model_c_underprediction_ge4']=bmc.model_c_residual_log2>=4
bmc['bmc_at_underprediction_ge2']=bmc.bmc_at_residual_log2>=2
bmc['bmc_at_underprediction_ge4']=bmc.bmc_at_residual_log2>=4
bmc['interval_width_log2']=bmc.bmc_at_upper95_log2-bmc.bmc_at_lower95_log2
if (bmc.interval_width_log2<=0).any():raise ValueError('Invalid Bayesian predictive intervals.')

def summarise(tab):
    mc=tab.model_c_residual_log2.to_numpy()
    bn=tab.bmc_at_residual_log2.to_numpy()
    return pd.Series({
        'n_test':len(tab),'n_pathogens':tab.biosample.nunique(),
        'model_c_MAE_log2':np.abs(mc).mean(),
        'bmc_at_MAE_log2':np.abs(bn).mean(),
        'model_c_RMSE_log2':np.sqrt(np.mean(mc**2)),
        'bmc_at_RMSE_log2':np.sqrt(np.mean(bn**2)),
        'interval_95pct_coverage':tab.covered_95pct.mean(),
        'n_above_upper95':int(tab.above_upper95.sum()),
        'n_below_lower95':int(tab.below_lower95.sum()),
        'pct_above_upper95':100*tab.above_upper95.mean(),
        'mean_interval_width_log2':tab.interval_width_log2.mean(),
        'n_model_c_underprediction_ge2':int(tab.model_c_underprediction_ge2.sum()),
        'n_model_c_underprediction_ge4':int(tab.model_c_underprediction_ge4.sum()),
        'n_bmc_at_underprediction_ge2':int(tab.bmc_at_underprediction_ge2.sum()),
        'n_bmc_at_underprediction_ge4':int(tab.bmc_at_underprediction_ge4.sum()),
    })

overall_summary=summarise(bmc).to_frame().T
antibiotic_summary=pd.DataFrame([{'antibiotic':name,**summarise(part).to_dict()}
                                 for name,part in bmc.groupby('antibiotic',sort=False)])
# Ensure numerical tables remain correctly typed and sorted by upper-limit misses.
antibiotic_summary=antibiotic_summary.sort_values(['n_above_upper95','interval_95pct_coverage'],ascending=[False,True]).reset_index(drop=True)
print('Overall held-out performance (50,460 observations):')
display(overall_summary.round(4))
print('Antibiotic-specific held-out performance (26 antibiotics):')
display(antibiotic_summary.round(4))
print('Reminder: coverage is a predictive interval metric; no universal resistance threshold is used.')


In [ ]:
#@title Cell 43.6 — Figures 1 and 2: coverage and upper-tail misses across 26 antibiotics
plot_table=antibiotic_summary.sort_values('interval_95pct_coverage',ascending=True)
fig,ax=plt.subplots(figsize=(11,11))
pos=np.arange(len(plot_table))
ax.barh(pos,100*plot_table.interval_95pct_coverage,color='C0',alpha=.82)
ax.axvline(95,color='black',ls='--',lw=1.5,label='Nominal 95% coverage')
ax.set_yticks(pos,plot_table.antibiotic)
ax.set_xlim(0,103)
ax.set_xlabel('Held-out observations within 95% BMC-AT predictive interval (%)')
ax.set_title('Figure 1 — BMC-AT coverage across 26 antibiotics')
ax.legend(loc='lower right')
ax.grid(axis='x',alpha=.2)
ax.set_axisbelow(True)
for y0,val,n in zip(pos,plot_table.interval_95pct_coverage,plot_table.n_test):
    ax.text(min(100*val+0.7,100.1),y0,f'n={int(n)}',va='center',fontsize=8)
fig.tight_layout()
fig.savefig(OUT/'43_Figure1_26_Antibiotic_Coverage.png',dpi=180,bbox_inches='tight')
plt.show()

miss_table=antibiotic_summary.sort_values('n_above_upper95',ascending=True)
fig,ax=plt.subplots(figsize=(11,11))
pos=np.arange(len(miss_table))
ax.barh(pos,miss_table.n_above_upper95,color='C1',alpha=.85)
ax.set_yticks(pos,miss_table.antibiotic)
ax.set_xlabel('Count: observed log₂(MIC) above BMC-AT upper 95% predictive limit')
ax.set_title('Figure 2 — Upper-tail misses among independent held-out predictions')
ax.grid(axis='x',alpha=.2)
ax.set_axisbelow(True)
maxmiss=int(miss_table.n_above_upper95.max())
ax.set_xlim(0,max(5,maxmiss*1.25+1))
for y0,count,pct in zip(pos,miss_table.n_above_upper95,miss_table.pct_above_upper95):
    ax.text(int(count)+max(0.2,maxmiss*.012),y0,f'{int(count)} ({pct:.1f}%)',va='center',fontsize=8)
fig.tight_layout()
fig.savefig(OUT/'43_Figure2_26_Antibiotic_Upper_Misses.png',dpi=180,bbox_inches='tight')
plt.show()


In [ ]:
#@title Cell 43.7 — Figure 3: individual held-out pathogens for four focus antibiotics
FOCUS_ANTIBIOTICS=['ceftazidime','cefepime','ciprofloxacin','meropenem']
FOCUS_N=25

for ab in FOCUS_ANTIBIOTICS:
    d=bmc.loc[bmc.antibiotic.eq(ab)].copy()
    if d.empty: raise ValueError(f'No independent held-out observations for {ab}.')
    # Highlight the 25 largest *original Model C* underpredictions, ensuring fixed plots
    # across BMC-N / T / AT in the three-way comparison in NB43.
    d=d.sort_values('model_c_residual_log2',ascending=False).head(FOCUS_N)
    yrows=np.arange(len(d))
    fig,ax=plt.subplots(figsize=(12,max(6,0.35*len(d)+2)))
    ax.hlines(yrows,d.bmc_at_lower95_log2,d.bmc_at_upper95_log2,color='C0',lw=2.5,
              label='BMC-AT 95% predictive interval')
    ax.scatter(d.bmc_at_median_log2,yrows,c='C0',marker='o',s=27,zorder=3,label='BMC-AT predictive median')
    ax.scatter(d.observed_log2_mic,yrows,c='C3',marker='x',s=47,lw=1.8,zorder=4,label='Observed log₂(MIC)')
    ax.set_yticks(yrows,d.biosample)
    ax.invert_yaxis()
    ax.set_xlabel('log₂(MIC), MIC in mg/L')
    ax.set_ylabel('BioSample')
    ax.set_title(f'Figure 3 — {ab}: {len(d)} largest underprediction cases (held-out)')
    ax.grid(axis='x',alpha=.23)
    ax.set_axisbelow(True)
    ax.legend(loc='lower right',fontsize=9)
    fig.tight_layout()
    fname=f'43_Figure3_Intervals_{ab.replace("-","_").replace(" ","_")}.png'
    fig.savefig(OUT/fname,dpi=180,bbox_inches='tight')
    plt.show()
    print(ab,':',int(d.above_upper95.sum()),'of plotted 25 above the upper 95% limit;',
          int(bmc.loc[bmc.antibiotic.eq(ab),'above_upper95'].sum()),'above among all held-out observations.')


In [ ]:
#@title Cell 43.8 — Save results and final validation audit
PRED_OUT=OUT/'43_BMC_AT_independent_outer_test_predictions.csv.gz'
POST_OUT=OUT/'43_BMC_AT_training_only_posterior_by_antibiotic_and_fold.csv'
ANT_OUT=OUT/'43_BMC_AT_26_antibiotic_validation_summary.csv'
ALL_OUT=OUT/'43_BMC_AT_overall_validation_summary.csv'
AUDIT_OUT=OUT/'43_BMC_AT_validation_audit.txt'

bmc.to_csv(PRED_OUT,index=False,compression='gzip')
posterior_summary.to_csv(POST_OUT,index=False)
antibiotic_summary.to_csv(ANT_OUT,index=False)
overall_summary.to_csv(ALL_OUT,index=False)

checks={
    'total_held_out_rows':len(bmc),
    'held_out_biosamples':int(bmc.biosample.nunique()),
    'antibiotics':int(bmc.antibiotic.nunique()),
    'model_fits':len(posterior_summary),
    'calibration_records_across_five_overlapping_outer_training_folds':len(calib),
    'distinct_independent_held_out_records':int(bmc.interaction_row.nunique()),
    'all_folds_pathogen_disjoint':True,
    'minimum_importance_ess':float(posterior_summary.importance_ess.min()),
    'coverage':float(bmc.covered_95pct.mean()),
    'n_upper_misses':int(bmc.above_upper95.sum()),
    'n_lower_misses':int(bmc.below_lower95.sum()),
}
if (checks['total_held_out_rows'],checks['held_out_biosamples'],checks['antibiotics'],checks['model_fits'])!=(50460,9058,26,130):
    raise ValueError('Final BMC-AT data size checks failed.')
AUDIT_OUT.write_text('\n'.join(f'{k}: {v}' for k,v in checks.items())+'\n',encoding='utf-8')
print('NB43 completed successfully. All reported BMC-AT metrics use independent held-out MIC observations.')
for f in [PRED_OUT,POST_OUT,ANT_OUT,ALL_OUT,AUDIT_OUT]:print('Saved:',f)
print('Saved six PNG figures in:',OUT)
print('Next: run Cell 43.9 to compare BMC-N, BMC-T and BMC-AT.')


In [ ]:
#@title Cell 43.9 — Compare BMC-N, BMC-T, and BMC-AT using the same 50,460 held-out observations
# Requires completed NB41 (N) and NB42 (T); this cell never refits or alters them.
N_NAME='41_BMC_N_independent_outer_test_predictions.csv.gz'
T_NAME='42_BMC_T_independent_outer_test_predictions.csv.gz'
N_PATH=ROOT/'03_Results'/'41_BMC_N_Pathogen_Out_Validation'/N_NAME
T_PATH=ROOT/'03_Results'/'42_BMC_T_Pathogen_Out_Validation'/T_NAME
if not N_PATH.is_file():N_PATH=get_file(N_NAME)
if not T_PATH.is_file():T_PATH=get_file(T_NAME)
N_TAB=pd.read_csv(N_PATH,compression='gzip')
T_TAB=pd.read_csv(T_PATH,compression='gzip')
A_TAB=bmc.copy()

key=['interaction_row','biosample','antibiotic','outer_fold','observed_log2_mic','model_c_outer_test_predicted_log2_mic']
for name,tab in [('N',N_TAB),('T',T_TAB),('AT',A_TAB)]:
    assert len(tab)==50460 and tab.interaction_row.is_unique and tab.antibiotic.nunique()==26, f'{name} invalid observation set'
ref=A_TAB.set_index('interaction_row')[key[1:]].sort_index()
for name,tab in [('N',N_TAB),('T',T_TAB)]:
    ct=tab.set_index('interaction_row')[key[1:]].sort_index()
    if not np.array_equal(ref[['biosample','antibiotic','outer_fold']].to_numpy(),
                          ct[['biosample','antibiotic','outer_fold']].to_numpy()):
        raise ValueError(f'BMC-{name} pathogen and fold keys differ from BMC-AT.')
    if not np.allclose(ref[['observed_log2_mic','model_c_outer_test_predicted_log2_mic']],
                       ct[['observed_log2_mic','model_c_outer_test_predicted_log2_mic']],rtol=0,atol=1e-9):
        raise ValueError(f'BMC-{name} MIC records differ from BMC-AT.')

metrics=[]
for tag,tab in [('N',N_TAB),('T',T_TAB),('AT',A_TAB)]:
    pre=f'bmc_{tag.lower()}_'
    for ab,g in [('ALL',tab)]+list(tab.groupby('antibiotic')):
        y=g.observed_log2_mic
        low=g[pre+'lower95_log2'];high=g[pre+'upper95_log2']
        pred=g[pre+'median_log2']
        metrics.append({'model':f'BMC-{tag}','antibiotic':ab,'n_test':len(g),
            'coverage_95pct':float(((y>=low)&(y<=high)).mean()),
            'n_above_upper95':int((y>high).sum()),'pct_above_upper95':100*float((y>high).mean()),
            'n_below_lower95':int((y<low).sum()),
            'mean_width_log2':float((high-low).mean()),
            'point_MAE_log2':float((y-pred).abs().mean())})
comparison=pd.DataFrame(metrics)
comparison.to_csv(OUT/'43_Three_Model_26_Antibiotic_Comparison.csv',index=False)
print('Direct comparison — same original Model C held-out observations (no common resistance breakpoint):')
display(comparison.loc[comparison.antibiotic.eq('ALL')].round(4))
print('Four focus antibiotics:')
display(comparison.loc[comparison.antibiotic.isin(FOCUS_ANTIBIOTICS)].round(4))

# Figure 4: All 26 antibiotic predictive coverage, same antibiotic order in all three models.
order=(comparison.loc[(comparison.model=='BMC-N')&(comparison.antibiotic!='ALL')]
       .sort_values('coverage_95pct').antibiotic.tolist())
Y=np.arange(26)
fig,ax=plt.subplots(figsize=(12,11))
for offset,tag,col in [(-.22,'N','C0'),(0,'T','C1'),(.22,'AT','C2')]:
    d=comparison.loc[(comparison.model==f'BMC-{tag}')].set_index('antibiotic').loc[order]
    ax.plot(d.coverage_95pct.to_numpy()*100,Y+offset,'.',ms=9,color=col,label=f'BMC-{tag}')
ax.axvline(95,color='black',lw=1.3,ls='--',label='Nominal 95%')
ax.set(yticks=Y,yticklabels=order,xlim=(75,101),xlabel='Held-out 95% interval coverage (%)',
       title='Figure 4 — Three Bayesian likelihoods: coverage by antibiotic')
ax.grid(axis='x',alpha=.2);ax.legend(loc='lower right');fig.tight_layout()
fig.savefig(OUT/'43_Figure4_Three_Model_Coverage.png',dpi=180,bbox_inches='tight');plt.show()

# Figure 5: per-antibiotic % above upper 95% limit (comparable despite different sample counts).
fig,ax=plt.subplots(figsize=(12,11))
for offset,tag,col in [(-.22,'N','C0'),(0,'T','C1'),(.22,'AT','C2')]:
    d=comparison.loc[(comparison.model==f'BMC-{tag}')].set_index('antibiotic').loc[order]
    ax.plot(d.pct_above_upper95.to_numpy(),Y+offset,'.',ms=9,color=col,label=f'BMC-{tag}')
ax.axvline(2.5,color='black',lw=1.3,ls='--',label='Symmetric nominal upper-tail 2.5%')
ax.set(yticks=Y,yticklabels=order,xlabel='Held-out observed MIC above 97.5% prediction limit (%)',
       title='Figure 5 — Upper-tail misses by antibiotic')
ax.grid(axis='x',alpha=.2);ax.legend(loc='lower right');fig.tight_layout()
fig.savefig(OUT/'43_Figure5_Three_Model_Upper_Tail.png',dpi=180,bbox_inches='tight');plt.show()

# Figure 6: each row is an identical held-out BioSample, ordered by original Model C underprediction.
# Three side-by-side panels per focus antibiotic, sharing the x range and BioSample order.
T_IDX=T_TAB.set_index('interaction_row')
N_IDX=N_TAB.set_index('interaction_row')
for ab in FOCUS_ANTIBIOTICS:
    sub=A_TAB.loc[A_TAB.antibiotic.eq(ab)].copy()
    sub['model_c_residual']=sub.observed_log2_mic-sub.model_c_outer_test_predicted_log2_mic
    sub=sub.sort_values('model_c_residual',ascending=False).head(FOCUS_N)
    ids=sub.interaction_row.to_numpy()
    inds=np.arange(len(ids))
    fig,axs=plt.subplots(1,3,figsize=(20,10),sharey=True)
    alls=[]
    for ax,(tag,tab,col) in zip(axs,[('N',N_IDX,'C0'),('T',T_IDX,'C1'),('AT',A_TAB.set_index('interaction_row'),'C2')]):
        d=tab.loc[ids]
        p=f'bmc_{tag.lower()}_'
        ax.hlines(inds,d[p+'lower95_log2'],d[p+'upper95_log2'],color=col,lw=2.2)
        ax.scatter(d[p+'median_log2'],inds,color=col,marker='o',s=22,zorder=3)
        ax.scatter(d.observed_log2_mic,inds,color='C3',marker='x',s=42,zorder=4)
        alls.extend([d[p+'lower95_log2'].min(),d[p+'upper95_log2'].max(),d.observed_log2_mic.min(),d.observed_log2_mic.max()])
        ax.set_title(f'BMC-{tag}')
        ax.set_xlabel('log₂(MIC), mg/L')
        ax.grid(axis='x',alpha=.2)
    axs[0].invert_yaxis()
    xmin,xmax=min(alls),max(alls)
    for ax in axs:
        ax.set_xlim(xmin-0.5,xmax+0.5)
        ax.set_yticks(inds)
        ax.set_yticklabels(sub.biosample.to_list(),fontsize=7)
    fig.suptitle(f'Figure 6 — {ab}: same {len(sub)} largest Model C underprediction cases')
    fig.tight_layout(rect=(0,0,1,.97))
    path=OUT/f'43_Figure6_Three_Model_Intervals_{ab.replace("-","_").replace(" ","_")}.png'
    fig.savefig(path,dpi=160,bbox_inches='tight');plt.show()
print('Three-model comparison saved in',OUT)
